In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [2]:
property_gdf = gpd.read_parquet("../../data/processed/nsw_property/property.parquet")
heritage_gdf = gpd.read_parquet("../../data/processed/nsw_heritage/heritage.parquet")
site_features = gpd.read_parquet("../../data/processed/site_features/property_site_features_zoning_sample.parquet")

In [3]:
print("Property rows:", len(property_gdf))
print("Heritage rows:", len(heritage_gdf))
print("Site rows:", len(site_features))

print("Property CRS:", property_gdf.crs)
print("Heritage CRS:", heritage_gdf.crs)
print("Site CRS:", site_features.crs)

Property rows: 4198396
Heritage rows: 40422
Site rows: 49950
Property CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviat

In [4]:
property_main = property_gdf[property_gdf["propertytype"] == 1].copy()
print(len(property_main))

4193334


In [5]:
heritage_keep = heritage_gdf[
    [
        "OBJECTID",
        "EPI_NAME",
        "LGA_NAME",
        "LAY_CLASS",
        "H_ID",
        "H_NAME",
        "SIG",
        "EPI_TYPE",
        "geometry",
    ]
].copy()

if property_main.crs != heritage_keep.crs:
    property_main = property_main.to_crs(heritage_keep.crs)

sample_rids = set(site_features["RID"].unique())
property_sample = property_main[property_main["RID"].isin(sample_rids)].copy()
print(len(property_sample))

49950


In [6]:
joined = gpd.sjoin(
    property_sample,
    heritage_keep,
    how="left",
    predicate="intersects"
)

In [7]:
print("Joined rows:", len(joined))
joined[
    ["RID", "address", "LAY_CLASS", "H_NAME", "SIG", "EPI_NAME", "LGA_NAME"]
].head(10)

Joined rows: 52408


,RID,address,LAY_CLASS,H_NAME,SIG,EPI_NAME,LGA_NAME
173,260,44 NORTHCOTE STREET ABERDARE,NaN,NaN,NaN,NaN,NaN
217,319,10 MEREWETHER CLOSE NORTH ROTHBURY,NaN,NaN,NaN,NaN,NaN
221,327,112 MATHIESON STREET BELLBIRD HEIGHTS,NaN,NaN,NaN,NaN,NaN
232,338,584 WOLLOMBI ROAD BELLBIRD,NaN,NaN,NaN,NaN,NaN
251,359,41 RUSSELL STREET BRANXTON,NaN,NaN,NaN,NaN,NaN
285,400,14 FOSTER STREET CESSNOCK,NaN,NaN,NaN,NaN,NaN
327,456,13 WILLIAM STREET CESSNOCK,NaN,NaN,NaN,NaN,NaN
415,606,761 WATAGAN CREEK ROAD WATAGAN,NaN,NaN,NaN,NaN,NaN
616,913,4 CORAL CRESCENT PEARL BEACH,NaN,NaN,NaN,NaN,NaN
652,951,65 DONALD AVENUE UMINA BEACH,NaN,NaN,NaN,NaN,NaN


In [8]:
joined["LAY_CLASS"].isna().mean()

np.float64(0.8388413982598076)

In [9]:
joined["LAY_CLASS"].value_counts(dropna=False).head(20)

LAY_CLASS
NaN                                          43962
Item - General                                4486
Conservation Area - General                   3445
Item - Landscape                               239
Item - Archaeological                          137
Conservation Area - Landscape                  103
Conservation Area - Aboriginal                  19
Aboriginal Place of Heritage Significance       13
Local Heritage - General                         2
Conservation Area - Archaeological               1
Item - Aboriginal                                1
Name: count, dtype: int64

In [10]:
joined["RID"].duplicated().mean()

np.float64(0.04690123645245001)

In [11]:
def collapse_heritage_group(group: pd.DataFrame) -> pd.Series:
    lay_nonnull = group["LAY_CLASS"].dropna()
    sig_nonnull = group["SIG"].dropna()
    hname_nonnull = group["H_NAME"].dropna()

    heritage_classes = sorted(lay_nonnull.astype(str).unique().tolist()) if len(lay_nonnull) else []
    heritage_names = sorted(hname_nonnull.astype(str).unique().tolist()) if len(hname_nonnull) else []

    sig_order = {
        "Local": 1,
        "State": 2,
        "National": 3,
    }

    if len(sig_nonnull):
        sig_candidates = sig_nonnull.astype(str).tolist()
        max_sig = max(sig_candidates, key=lambda x: sig_order.get(x, 0))
    else:
        max_sig = pd.NA

    return pd.Series(
        {
            "heritage_flag": int(group["LAY_CLASS"].notna().any()),
            "heritage_hit_count": int(group["LAY_CLASS"].notna().sum()),
            "heritage_class_count": int(lay_nonnull.nunique()),
            "heritage_classes": "|".join(heritage_classes),
            "heritage_names": "|".join(heritage_names),
            "heritage_max_significance": max_sig,
        }
    )

In [12]:
heritage_features = (
    joined.groupby("RID", as_index=False)
    .apply(collapse_heritage_group)
    .reset_index(drop=True)
)

print(len(heritage_features))
heritage_features.head()

49950


/var/folders/kc/4swzx7w979z6w9js5c61gt7h0000gn/T/ipykernel_39871/4222738825.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(collapse_heritage_group)


,RID,heritage_flag,heritage_hit_count,heritage_class_count,heritage_classes,heritage_names,heritage_max_significance
0,260,0,0,0,,,<NA>
1,319,0,0,0,,,<NA>
2,327,0,0,0,,,<NA>
3,338,0,0,0,,,<NA>
4,359,0,0,0,,,<NA>


In [13]:
site_with_heritage = site_features.merge(
    heritage_features,
    on="RID",
    how="left",
)

print(len(site_with_heritage))
site_with_heritage.head()

49950


,RID,gurasid,propertytype,valnetpropertystatus,valnetpropertytype,dissolveparcelcount,valnetlotcount,propid,superlot,address,...,has_zoning,is_large_lot_outlier,is_tiny_lot_outlier,log_lot_size_proxy_sqm,heritage_flag,heritage_hit_count,heritage_class_count,heritage_classes,heritage_names,heritage_max_significance
0,260,49494103,1,2.0,2.0,1,1,1341,N,44 NORTHCOTE STREET ABERDARE,...,1,0,0,7.272967,0,0,0,,,<NA>
1,319,49494159,1,2.0,2.0,1,1,3169,N,10 MEREWETHER CLOSE NORTH ROTHBURY,...,1,0,0,10.146831,0,0,0,,,<NA>
2,327,49494166,1,2.0,2.0,1,1,3682,N,112 MATHIESON STREET BELLBIRD HEIGHTS,...,1,0,0,7.030290,0,0,0,,,<NA>
3,338,49494176,1,2.0,2.0,1,1,3958,N,584 WOLLOMBI ROAD BELLBIRD,...,1,0,0,6.896501,0,0,0,,,<NA>
4,359,49494197,1,2.0,2.0,1,1,4739,N,41 RUSSELL STREET BRANXTON,...,1,0,0,7.206756,0,0,0,,,<NA>


In [14]:
site_with_heritage["heritage_flag"] = site_with_heritage["heritage_flag"].fillna(0).astype(int)
site_with_heritage["heritage_hit_count"] = site_with_heritage["heritage_hit_count"].fillna(0).astype(int)
site_with_heritage["heritage_class_count"] = site_with_heritage["heritage_class_count"].fillna(0).astype(int)

In [15]:
print(site_with_heritage["heritage_flag"].value_counts(normalize=True, dropna=False))
print(site_with_heritage["heritage_max_significance"].value_counts(dropna=False))
print(site_with_heritage["heritage_class_count"].describe())

heritage_flag
0    0.88012
1    0.11988
Name: proportion, dtype: float64
heritage_max_significance
<NA>     43962
Local     5585
State      403
Name: count, dtype: int64
count    49950.000000
mean         0.142122
std          0.410333
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          4.000000
Name: heritage_class_count, dtype: float64


In [16]:
print(joined["LAY_CLASS"].isna().mean())
print(joined["RID"].duplicated().mean())
print(site_with_heritage["heritage_flag"].value_counts(normalize=True, dropna=False))
print(site_with_heritage["heritage_max_significance"].value_counts(dropna=False))

0.8388413982598076
0.04690123645245001
heritage_flag
0    0.88012
1    0.11988
Name: proportion, dtype: float64
heritage_max_significance
<NA>     43962
Local     5585
State      403
Name: count, dtype: int64


In [17]:
Path("../../data/processed/site_features").mkdir(parents=True, exist_ok=True)

site_with_heritage.to_parquet(
    "../../data/processed/site_features/property_site_features_zoning_heritage_sample.parquet",
    index=False,
)